In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
results = pd.read_csv('../data/raw/results.csv')
print(results.shape)
results.head()

(49287, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [6]:
print("Data range:", results['date'].min(), "to", results['date'].max())
print("Unique teams:", results['home_team'].nunique())
print("Null values:\n", results.isnull().sum())
print("\nTournament types:")
print(results['tournament'].value_counts().head(15))

Data range: 1872-11-30 to 2026-06-27
Unique teams: 325
Null values:
 date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

Tournament types:
tournament
Friendly                                18252
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Name: count, dtype: int64


In [10]:
# Check what the null-score rows look like
null_matches = results[results['home_score'].isnull()]
print(f"Matches with no score: {len(null_matches)}")
print(null_matches[['date', 'home_team', 'away_team', 'tournament']].head(10))

Matches with no score: 72
             date      home_team               away_team      tournament
49215  2026-06-11         Mexico            South Africa  FIFA World Cup
49216  2026-06-11    South Korea          Czech Republic  FIFA World Cup
49217  2026-06-12         Canada  Bosnia and Herzegovina  FIFA World Cup
49218  2026-06-12  United States                Paraguay  FIFA World Cup
49219  2026-06-13          Qatar             Switzerland  FIFA World Cup
49220  2026-06-13         Brazil                 Morocco  FIFA World Cup
49221  2026-06-13          Haiti                Scotland  FIFA World Cup
49222  2026-06-13      Australia                  Turkey  FIFA World Cup
49223  2026-06-14        Germany                 Curaçao  FIFA World Cup
49224  2026-06-14    Ivory Coast                 Ecuador  FIFA World Cup


In [8]:
# Check the most recent matches with actual scores
recent = results[results['home_score'].notna()].sort_values('date', ascending=False)
print(recent[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].head(10))

             date    home_team    away_team  home_score  away_score  \
49214  2026-03-31   Kazakhstan      Comoros         1.0         0.0   
49193  2026-03-31       Norway  Switzerland         0.0         0.0   
49191  2026-03-31  Netherlands      Ecuador         1.0         1.0   
49189  2026-03-31   Montenegro     Slovenia         2.0         3.0   
49188  2026-03-31       Mexico      Belgium         1.0         1.0   
49187  2026-03-31     Botswana       Malawi         1.0         0.0   
49186  2026-03-31      Liberia        Libya         2.0         2.0   
49185  2026-03-31       Jordan      Nigeria         2.0         2.0   
49184  2026-03-31  Ivory Coast     Scotland         1.0         0.0   
49183  2026-03-31      Hungary       Greece         0.0         0.0   

                                 tournament  
49214                           FIFA Series  
49193                              Friendly  
49191                              Friendly  
49189                             

In [12]:
# Seperate future fixtures from training data. At the time that we retrieved the data, May 27, it also included data from the world cup, data that is not valid yet
wc_fixtures = results[results['home_score'].isnull()].copy()
results_clean = results[results['home_score'].notna()].copy()

print(f"Training matches: {len(results_clean)}")
print(f"WC fixtures to predict: {len(wc_fixtures)}")
print(f"\nFirst WC fixture: {wc_fixtures.iloc[0]['date']} - {wc_fixtures.iloc[0]['home_team']} vs {wc_fixtures.iloc[0]['away_team']}")

Training matches: 49215
WC fixtures to predict: 72

First WC fixture: 2026-06-11 - Mexico vs South Africa


In [14]:
# How does data volume look by decade?
results_clean['year'] = pd.to_datetime(results_clean['date']).dt.year
results_clean['decade'] = (results_clean['year'] // 10) * 10

print(results_clean.groupby('decade').size())

decade
1870      13
1880      55
1890      59
1900     137
1910     330
1920     828
1930    1079
1940     833
1950    1651
1960    2971
1970    4133
1980    5025
1990    6944
2000    9525
2010    9756
2020    5876
dtype: int64


In [16]:
# How many World Cup matches specifically do we have?
wc_only = results_clean[results_clean['tournament'] == 'FIFA World Cup']
print(f"Total World Cup matches in training data: {len(wc_only)}")
print(f"Years covered: {wc_only['year'].min()} to {wc_only['year'].max()}")

Total World Cup matches in training data: 964
Years covered: 1930 to 2022


In [17]:
# What's the distribution of results?
results_clean['result'] = results_clean.apply(
    lambda row: 'Home Win' if row['home_score'] > row['away_score']
    else ('Away Win' if row['home_score'] < row['away_score'] else 'Draw'),
    axis=1
)

print(results_clean['result'].value_counts())
print("\nAs percentages:")
print(results_clean['result'].value_counts(normalize=True).round(3) * 100)

result
Home Win    24106
Away Win    13912
Draw        11197
Name: count, dtype: int64

As percentages:
result
Home Win    49.0
Away Win    28.3
Draw        22.8
Name: proportion, dtype: float64


In [18]:
neutral_results = results_clean[results_clean['neutral'] == True]
print(f"Total neutral matches: {len(neutral_results)}")
print("\nNeutral match outcomes:")
print(neutral_results['result'].value_counts(normalize=True).round(3) * 100)


Total neutral matches: 12976

Neutral match outcomes:
result
Home Win    44.1
Away Win    33.4
Draw        22.4
Name: proportion, dtype: float64


In [19]:
rankings = pd.read_csv('../data/raw/fifa_ranking-2024-06-20.csv')

# Always check shape first - tells you rows x columns at a glance
print("Shape:", rankings.shape)

# Column names tell you what features are available
print("Columns:", rankings.columns.tolist())


# Head shows you what the actual data looks like
rankings.head(10)

Shape: (67472, 8)
Columns: ['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31
5,29.0,Yugoslavia,YUG,39.0,0.0,29,UEFA,1992-12-31
6,28.0,Wales,WAL,40.0,0.0,28,UEFA,1992-12-31
7,27.0,Côte d'Ivoire,CIV,41.0,0.0,27,CAF,1992-12-31
8,34.0,Austria,AUT,38.0,0.0,34,UEFA,1992-12-31
9,26.0,Bulgaria,BUL,41.0,0.0,26,UEFA,1992-12-31


In [22]:
# How many unique ranking dates do we have?
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])
print("Ranking history:", rankings['rank_date'].min(), "to", rankings['rank_date'].max())
print("Unique update dates:", rankings['rank_date'].nunique())
print("Unique teams ever ranked:", rankings['country_full'].nunique())

# What does one team's ranking history look like over time?
brazil_rankings = rankings[rankings['country_full'] == 'Brazil'].sort_values('rank_date')
print("\nBrazil ranking history (last 5 entries):")
print(brazil_rankings[['rank_date', 'rank', 'total_points']].tail())

mexico_rankings = rankings[rankings['country_full'] == 'Mexico'].sort_values('rank_date')
print("\nMexico ranking history (last 10 entries):")
print(mexico_rankings[['rank_date', 'rank', 'total_points']].tail(10))


Ranking history: 1992-12-31 00:00:00 to 2024-06-20 00:00:00
Unique update dates: 333
Unique teams ever ranked: 216

Brazil ranking history (last 5 entries):
       rank_date  rank  total_points
66488 2023-11-30   5.0       1784.09
66699 2023-12-21   5.0       1784.09
66981 2024-02-15   5.0       1784.09
67054 2024-04-04   5.0       1788.65
67332 2024-06-20   4.0       1791.85

Mexico ranking history (last 10 entries):
       rank_date  rank  total_points
65373 2023-04-06  15.0       1631.87
65582 2023-06-29  14.0       1639.19
65862 2023-07-20  12.0       1665.59
66148 2023-09-21  12.0       1661.46
66269 2023-10-26  12.0       1663.94
66479 2023-11-30  14.0       1655.21
66689 2023-12-21  15.0       1652.70
66991 2024-02-15  15.0       1652.70
67061 2024-04-04  14.0       1661.11
67322 2024-06-20  15.0       1652.33


In [23]:
players = pd.read_csv('../data/raw/ea_fc26_players.csv')
print("Shape:", players.shape)
print("Columns:", players.columns.tolist())
players.head(5)

Shape: (16228, 60)
Columns: ['id', 'rank', 'overallRating', 'firstName', 'lastName', 'commonName', 'birthdate', 'height', 'weight', 'skillMoves', 'weakFootAbility', 'preferredFoot', 'position', 'positionType', 'alternatePositions', 'nationality', 'team', 'leagueName', 'playStyles', 'playStylesPlus', 'pac', 'sho', 'pas', 'dri', 'def', 'phy', 'acceleration', 'sprintSpeed', 'finishing', 'shotPower', 'longShots', 'volleys', 'penalties', 'positioning', 'shortPassing', 'longPassing', 'curve', 'freeKickAccuracy', 'crossing', 'vision', 'dribbling', 'ballControl', 'agility', 'balance', 'reactions', 'composure', 'interceptions', 'defensiveAwareness', 'standingTackle', 'slidingTackle', 'headingAccuracy', 'aggression', 'jumping', 'stamina', 'strength', 'gkDiving', 'gkHandling', 'gkKicking', 'gkPositioning', 'gkReflexes']


,id,rank,overallRating,firstName,lastName,commonName,birthdate,height,weight,skillMoves,weakFootAbility,preferredFoot,position,positionType,alternatePositions,nationality,team,leagueName,playStyles,playStylesPlus,pac,sho,pas,dri,def,phy,acceleration,sprintSpeed,finishing,shotPower,longShots,volleys,penalties,positioning,shortPassing,longPassing,curve,freeKickAccuracy,crossing,vision,dribbling,ballControl,agility,balance,reactions,composure,interceptions,defensiveAwareness,standingTackle,slidingTackle,headingAccuracy,aggression,jumping,stamina,strength,gkDiving,gkHandling,gkKicking,gkPositioning,gkReflexes
0,209331,1,91,Mohamed,Salah,NaN,6/15/1992 12:00:00 AM,175,72,4,3,2,RM,Midfielder,RW,Egypt,Liverpool,Premier League,"Low Driven Shot,Gamechanger,Whipped Pass,Inven...",Finesse Shot+,89,88,86,90,45,76,88,89,94,83,78,83,88,93,88,81,88,69,86,89,90,90,86,91,94,93,55,38,43,41,59,63,79,88,75,14,14,9,11,14
1,231747,3,91,Kylian,Mbappé,NaN,12/20/1998 12:00:00 AM,182,75,5,4,1,ST,Attack,"LW,LM",France,Real Madrid,LALIGA EA SPORTS,"Finesse Shot,Acrobatic,Low Driven Shot,Gamecha...",Quick Step+,97,90,81,92,37,76,97,97,92,91,86,87,82,91,87,74,80,69,78,83,92,93,93,82,91,88,38,26,34,32,78,61,90,83,77,13,5,7,11,6
2,231443,5,90,Ousmane,Dembélé,NaN,5/15/1997 12:00:00 AM,178,67,5,5,2,ST,Attack,"RW,CAM",France,Paris SG,Ligue 1 McDonald's,"Low Driven Shot,Pinged Pass,Inventive,Trickste...",Rapid+,91,88,83,93,50,69,93,89,90,91,85,79,80,95,89,78,85,68,80,84,94,94,94,81,91,88,45,49,49,39,74,58,84,76,69,6,6,14,10,13
3,231866,6,90,Rodrigo,Hernández Cascante,Rodri,6/22/1996 12:00:00 AM,190,82,3,4,1,CDM,Midfielder,CM,Spain,Manchester City,Premier League,"Power Shot,Long Ball Pass,Aerial Fortress,Pres...",Tiki Taka+,65,80,86,84,86,85,65,65,74,92,89,71,62,76,93,91,86,64,76,84,84,90,66,67,93,93,84,88,87,82,81,85,83,91,83,10,10,7,14,8
4,203376,8,90,Virgil,van Dijk,NaN,7/8/1991 12:00:00 AM,193,92,2,3,1,CB,Defense,NaN,Holland,Liverpool,Premier League,"Precision Header,Pinged Pass,Jockey,Anticipate...",Intercept+,73,60,72,72,90,87,66,78,52,81,64,45,62,47,80,83,60,70,53,70,70,77,54,50,90,90,91,91,91,87,88,85,89,75,93,13,10,13,11,11


In [25]:
# Do we have FC26 players for all 48 World Cup teams???

# How many nationalities are in the FC26 data?
print("Unique nationalities:", players['nationality'].nunique())

# Check a few key WC teams
wc_teams = ['Brazil', 'France', 'Argentina', 'England', 'Mexico', 'Spain', 'Germany', 'Portugal', 'Japan', 'Morocco']

for team in wc_teams:
    count = len(players[players['nationality'] == team])
    top_player = players[players['nationality'] == team].nlargest(1, 'overallRating')[['firstName', 'lastName', 'overallRating']].values
    print(f"{team}: {count} players, best = {top_player}")

Unique nationalities: 156
Brazil: 343 players, best = [['Raphael' 'Dias Belloli' 89]]
France: 843 players, best = [['Kylian' 'Mbappé' 91]]
Argentina: 979 players, best = [['Lautaro' 'Martínez' 88]]
England: 1411 players, best = [['Jude' 'Bellingham' 90]]
Mexico: 27 players, best = [['Julian' 'Quiñones' 80]]
Spain: 843 players, best = [['Rodrigo' 'Hernández Cascante' 90]]
Germany: 1122 players, best = [['Joshua' 'Kimmich' 89]]
Portugal: 302 players, best = [['Vítor' 'Machado Ferreira' 89]]
Japan: 99 players, best = [['Kaoru' 'Mitoma' 82]]
Morocco: 121 players, best = [['Achraf' 'Hakimi' 89]]


In [26]:
# Extract all 48 WC teams from the fixtures we already loaded
home_teams = wc_fixtures['home_team'].unique()
away_teams = wc_fixtures['away_team'].unique()
wc_teams_48 = sorted(set(list(home_teams) + list(away_teams)))
print(f"Total WC teams: {len(wc_teams_48)}")

# Check FC26 coverage for every single WC team
coverage = []
for team in wc_teams_48:
    count = len(players[players['nationality'] == team])
    top_rating = players[players['nationality'] == team]['overallRating'].max() if count > 0 else 0
    coverage.append({'team': team, 'fc26_players': count, 'best_rating': top_rating})

coverage_df = pd.DataFrame(coverage).sort_values('fc26_players')
print("\nTeams with THIN coverage (under 20 players):")
print(coverage_df[coverage_df['fc26_players'] < 20].to_string(index=False))
print("\nTeams with ZERO players:")
print(coverage_df[coverage_df['fc26_players'] == 0].to_string(index=False))


Total WC teams: 48

Teams with THIN coverage (under 20 players):
        team  fc26_players  best_rating
 Ivory Coast             0            0
 South Korea             0            0
       Qatar             0            0
 Netherlands             0            0
    DR Congo             0            0
  Cape Verde             0            0
      Jordan             2           75
  Uzbekistan             4           77
        Iran             6           78
       Egypt             9           91
      Panama            11           76
     Curaçao            12           71
       Haiti            12           72
South Africa            14           73
        Iraq            15           72

Teams with ZERO players:
       team  fc26_players  best_rating
Ivory Coast             0            0
South Korea             0            0
      Qatar             0            0
Netherlands             0            0
   DR Congo             0            0
 Cape Verde             0          